# ReciLink Dummy Graph — Neo4j AuraDB
Loads 50 dummy recipes from `dummy_recipes.json` into Neo4j.
Creates Recipe nodes, Ingredient nodes, CONTAINS and CO_OCCURS relationships.

## Step 1 — Install Dependencies

In [ ]:
!pip install neo4j pandas

## Step 2 — Import Libraries

In [ ]:
import json
import pandas as pd
from neo4j import GraphDatabase

## Step 3 — Load Dummy Recipes

In [ ]:
with open('dummy_recipes.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f'Total recipes loaded: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Total recipes loaded: 57
Columns: ['title', 'description', 'cuisine', 'difficulty', 'prep_time', 'cook_time', 'servings', 'ingredients', 'steps', 'tags', 'image_url', 'source']


,title,description,cuisine,difficulty,prep_time,cook_time,servings,ingredients,steps,tags,image_url,source
0,Nasi Lemak,Malaysia's national dish â€” fragrant coconut ...,Malaysian,Medium,20,40,4,"[coconut milk, rice, pandan leaves, anchovies,...","[Wash rice and soak for 30 minutes, Cook rice ...","[halal, malaysian, breakfast, rice, spicy]",https://asianinspirations.com.au/wp-content/up...,dataset
1,Rendang Daging,A rich and aromatic slow-cooked Malaysian beef...,Malaysian,Hard,30,120,6,"[beef, coconut milk, lemongrass, galangal, chi...","[Blend chilli, onion, garlic, lemongrass and g...","[halal, malaysian, beef, spicy, slow-cook]",https://friedchillies.com/wp-content/uploads/2...,dataset
2,Char Kway Teow,A popular Malaysian stir-fried flat rice noodl...,Malaysian,Medium,15,15,2,"[flat rice noodles, prawns, Chinese sausage, e...","[Heat wok over high flame with oil, Fry garlic...","[halal, malaysian, noodles, seafood, stir-fry]",https://media.myresipi.com/2024/12/resepi-char...,dataset
3,Laksa Lemak,A rich and spicy coconut milk based noodle sou...,Malaysian,Hard,30,45,4,"[rice noodles, coconut milk, prawns, fish cake...","[Blend lemongrass, chilli, garlic, onion, shri...","[halal, malaysian, noodles, soup, spicy, seafood]",https://www.themealdb.com/images/media/meals/r...,dataset
4,Satay Ayam,Grilled skewered chicken marinated in turmeric...,Malaysian,Medium,60,20,4,"[chicken, lemongrass, turmeric, garlic, shallo...","[Blend marinade with lemongrass, turmeric, gar...","[halal, malaysian, chicken, grilled, peanut]",https://www.unileverfoodsolutions.com.my/dam/g...,dataset


## Step 4 — Connect to Neo4j AuraDB

In [ ]:
# Replace with your AuraDB credentials
NEO4J_URI      = 'neo4j+s://03461565.databases.neo4j.io'
NEO4J_USERNAME = '03461565'
NEO4J_PASSWORD = 'lZasvGUrxSMR3ToW1tsDgLL9P1Ttc7u9g01qiTt4sRY'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
with driver.session() as session:
    result = session.run('RETURN 1 AS test')
    print('Connection successful:', result.single()['test'])

Connection successful: 1


## Step 5 — Create Recipe Nodes

In [ ]:
def create_recipe_node(tx, title, cuisine, difficulty):
    tx.run("""
        MERGE (r:Recipe {name: $title})
        SET r.cuisine = $cuisine,
            r.difficulty = $difficulty
    """, title=title, cuisine=cuisine, difficulty=difficulty)

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(
            create_recipe_node,
            title=row['title'],
            cuisine=row['cuisine'],
            difficulty=row['difficulty']
        )

# Verify
with driver.session() as session:
    count = session.run('MATCH (r:Recipe) RETURN count(r) AS count').single()['count']
    print(f'✅ Recipe nodes created: {count}')

✅ Recipe nodes created: 57


## Step 6 — Create Ingredient Nodes + CONTAINS Relationships

In [ ]:
def create_ingredient_and_relationship(tx, recipe_title, ingredient):
    tx.run("""
        MERGE (i:Ingredient {name: $ingredient})
        WITH i
        MATCH (r:Recipe {name: $recipe_title})
        MERGE (r)-[:CONTAINS]->(i)
    """, ingredient=ingredient.lower().strip(), recipe_title=recipe_title)

with driver.session() as session:
    for _, row in df.iterrows():
        for ingredient in row['ingredients']:
            session.execute_write(
                create_ingredient_and_relationship,
                recipe_title=row['title'],
                ingredient=ingredient
            )

# Verify
with driver.session() as session:
    ing_count = session.run('MATCH (i:Ingredient) RETURN count(i) AS count').single()['count']
    rel_count = session.run('MATCH ()-[r:CONTAINS]->() RETURN count(r) AS count').single()['count']
    print(f'✅ Ingredient nodes created: {ing_count}')
    print(f'✅ CONTAINS relationships created: {rel_count}')

✅ Ingredient nodes created: 214
✅ CONTAINS relationships created: 635


## Step 7 — Create CO_OCCURS Relationships Between Ingredients

In [ ]:
def create_cooccurrence(tx, recipe_title):
    tx.run("""
        MATCH (r:Recipe {name: $title})-[:CONTAINS]->(i1:Ingredient)
        MATCH (r)-[:CONTAINS]->(i2:Ingredient)
        WHERE i1 <> i2
        MERGE (i1)-[c:CO_OCCURS]->(i2)
        ON CREATE SET c.frequency = 1
        ON MATCH SET c.frequency = c.frequency + 1
    """, title=recipe_title)

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(create_cooccurrence, recipe_title=row['title'])

# Verify
with driver.session() as session:
    co_count = session.run('MATCH ()-[r:CO_OCCURS]->() RETURN count(r) AS count').single()['count']
    print(f'✅ CO_OCCURS relationships created: {co_count}')

✅ CO_OCCURS relationships created: 4520


## Step 8 — Verify Full Graph Summary

In [ ]:
with driver.session() as session:
    recipes = session.run('MATCH (r:Recipe) RETURN count(r) AS count').single()['count']
    ingredients = session.run('MATCH (i:Ingredient) RETURN count(i) AS count').single()['count']
    contains = session.run('MATCH ()-[r:CONTAINS]->() RETURN count(r) AS count').single()['count']
    cooccurs = session.run('MATCH ()-[r:CO_OCCURS]->() RETURN count(r) AS count').single()['count']

    print('='*40)
    print('       RECILINK GRAPH SUMMARY')
    print('='*40)
    print(f'  Recipe nodes      : {recipes}')
    print(f'  Ingredient nodes  : {ingredients}')
    print(f'  CONTAINS rels     : {contains}')
    print(f'  CO_OCCURS rels    : {cooccurs}')
    print(f'  Total nodes       : {recipes + ingredients}')
    print(f'  Total rels        : {contains + cooccurs}')
    print('='*40)

       RECILINK GRAPH SUMMARY
  Recipe nodes      : 57
  Ingredient nodes  : 214
  CONTAINS rels     : 635
  CO_OCCURS rels    : 4520
  Total nodes       : 271
  Total rels        : 5155


## Step 9 — Test Recommendation Query

In [ ]:
test_ingredients = ['garlic', 'onion', 'coconut milk']

with driver.session() as session:
    result = session.run("""
        MATCH (r:Recipe)-[:CONTAINS]->(i:Ingredient)
        WHERE i.name IN $ingredients
        WITH r, count(i) AS matched
        ORDER BY matched DESC
        RETURN r.name AS recipe,
               r.difficulty AS difficulty,
               r.cuisine AS cuisine,
               matched
        LIMIT 5
    """, ingredients=test_ingredients)

    print(f'Top recipes for ingredients: {test_ingredients}')
    print('-'*50)
    for record in result:
        print(f"  {record['recipe']} ({record['cuisine']}, {record['difficulty']}) — {record['matched']} matched")

Top recipes for ingredients: ['garlic', 'onion', 'coconut milk']
--------------------------------------------------
  Vegetable Curry (General, Easy) — 3 matched
  Laksa Lemak (Malaysian, Hard) — 3 matched
  Rendang Daging (Malaysian, Hard) — 3 matched
  Roti Canai (Malaysian, Hard) — 3 matched
  Pumpkin Soup (General, Easy) — 3 matched


## Step 10 — Test Grocery Suggestion Query

In [ ]:
user_ingredients = ['garlic', 'onion', 'coconut milk']

with driver.session() as session:
    result = session.run("""
        MATCH (i1:Ingredient)-[c:CO_OCCURS]->(i2:Ingredient)
        WHERE i1.name IN $ingredients
        AND NOT i2.name IN $ingredients
        WITH i2.name AS suggested, sum(c.frequency) AS total
        ORDER BY total DESC
        RETURN suggested, total
        LIMIT 5
    """, ingredients=user_ingredients)

    print(f'Suggested groceries for: {user_ingredients}')
    print('-'*50)
    for record in result:
        print(f"  {record['suggested']} (co-occurrence: {record['total']})")

Suggested groceries for: ['garlic', 'onion', 'coconut milk']
--------------------------------------------------
  salt (co-occurrence: 48)
  black pepper (co-occurrence: 24)
  olive oil (co-occurrence: 20)
  egg (co-occurrence: 17)
  tomatoes (co-occurrence: 17)


## Step 11 — Close Driver

In [ ]:
driver.close()
print('✅ Driver closed. Graph loading complete!')

✅ Driver closed. Graph loading complete!
